In [1]:
import numpy as np
from jax import numpy as jnp

from blueprint.qubits import TunableTransmon, AnharmonicOscillator
from blueprint.devices import Device

In [2]:
label = "Q1"
charging_energy = 0.24193246329355922
josephson_energy = 16.861934403863096
charge_cutoff = 100
offset_charge = 0.0
dim = 3

transmon = TunableTransmon(
    label=label,
    charging_energy=charging_energy,
    josephson_energy=josephson_energy,
    offset_charge=offset_charge,
    charge_cutoff=charge_cutoff,
)
transmon.diagonalize(dim)

label = "Q2"
charging_energy = 0.23958546598194805
josephson_energy = 18.062900398155875
charge_cutoff = 100
offset_charge = 0.0
dim = 3

other_transmon = TunableTransmon(
    label=label,
    charging_energy=charging_energy,
    josephson_energy=josephson_energy,
    offset_charge=offset_charge,
    charge_cutoff=charge_cutoff,
)
other_transmon.diagonalize(dim)

# Comparing with the old implementation

In [3]:
EJ_GHz = transmon.max_josephson_energy
EC_GHz = transmon.charging_energy
ng = transmon.offset_charge
n_charge_states = transmon.charge_cutoff
dim = transmon.dim

dtype = jnp.complex64

charge_op = jnp.diag(jnp.arange(-1 * n_charge_states, n_charge_states + 1))
dim_charge = 2 * n_charge_states + 1
ones_offdiag = jnp.ones((dim_charge - 1,), dtype=dtype)
cosphi = jnp.diag(0.5 * ones_offdiag, k=1) + jnp.diag(0.5 * ones_offdiag, k=-1)
I = jnp.identity(dim_charge)
# H = 4 Ec (n - ng)^2 - Ej cos(phi)
H = 4 * EC_GHz * (charge_op - ng * I) @ (charge_op - ng * I) - EJ_GHz * cosphi
[D, V] = jnp.linalg.eigh(H)

eigvals = jnp.array(D[:dim] - D[0], dtype=dtype)

eigkets = V[:, :dim].T  # Shape (dim, dim_charge)
for iket, eket in enumerate(eigkets):
    index = jnp.argmax(abs(eket))
    eigkets.at[iket].set(eigkets[iket] * jnp.exp(-1j * jnp.angle(eket[index])))
eigkets = eigkets.T

H_diag = jnp.diag(eigvals)
# Truncate to `dim` levels only using this truncated eigenvectors matrix.
# trunc_V = V[:, :dim]
trunc_V = eigkets
n_diag = jnp.conj(trunc_V).T @ charge_op @ trunc_V
cosphi_diag = jnp.conj(trunc_V).T @ cosphi @ trunc_V

In [4]:
comp_charge_op = transmon._get_charge_op()
assert jnp.allclose(charge_op, comp_charge_op)

comp_cosphi_op = transmon._get_cosphi_op()
assert jnp.allclose(cosphi, comp_cosphi_op)

comp_hamil = transmon._get_hamiltonian()
assert jnp.allclose(H, comp_hamil)

comp_eig_vals, comp_eig_vecs = transmon.eigenstates()
assert jnp.allclose(D, comp_eig_vals)
assert jnp.allclose(V, comp_eig_vecs)

In [5]:
comp_transform = transmon._transform
jnp.allclose(trunc_V, comp_transform)

comp_diag_charge_op = transmon.get_charge_op()
assert jnp.allclose(n_diag, comp_diag_charge_op)

comp_diag_cosphi_op = transmon.get_cosphi_op()
assert jnp.allclose(cosphi_diag, comp_diag_cosphi_op)

comp_diag_hamil = transmon.get_hamiltonian()
assert jnp.allclose(H_diag, comp_diag_hamil)

In [6]:
device = Device((transmon, other_transmon))

In [7]:
device.add_capacative_coupling(
    qubit_labels=("Q1", "Q2"), coupler_label="G1", prefactor=0.0025
)

In [8]:
hamiltonian = device.get_hamiltonian()

Example of a flux-drive being applied to a transmon

In [9]:
label = "Q1"
charging_energy = 0.24193246329355922
josephson_energy = 16.861934403863096
charge_cutoff = 100
offset_charge = 0.0
dim = 3

transmon = TunableTransmon(
    label=label,
    charging_energy=charging_energy,
    josephson_energy=josephson_energy,
    offset_charge=offset_charge,
    charge_cutoff=charge_cutoff,
)

transmon.diagonalize(dim)

import math

def flux_pulse(
    time: float,
    max_voltage: float,
    flux_per_volt: float,
    modulation_freq: float,
    ramp_time: float,
    hold_time: float,
    std: float = 1.0,
) -> float:
    cos_term = math.cos(modulation_freq * time)

    sqrt2_std = math.sqrt(2) * std
    rise_term = math.erf((time - ramp_time) / sqrt2_std)
    fall_term = math.erf((time - ramp_time - hold_time) / sqrt2_std)
    input_voltage = 0.5 * max_voltage * cos_term * (rise_term - fall_term)
    applied_flux = input_voltage * flux_per_volt
    return applied_flux


transmon.add_flux_drive(label="cz_drive", flux_pulse=flux_pulse)

transmon.drives["cz_drive"].set_params(
    volt_per_flux=1/0.68, # in units of pi/phi_0
    voltage_amp=1.0, # in Voltage
    modulation_freq=0.564, # in GHz
    ramp_time=10.0, # in ns
    hold_time=20.0, # in ns
    std=5.0, # in ns
)

# At this point we should only have 'time' as the free parameter of the drive
print(transmon.drives["cz_drive"]. free_params)

['time']


In [11]:
time: float = 20
drive_hamil = transmon.get_drive_hamiltonian(time=time)
transmon.get_full_hamiltonian(time=time)

Array([[-2.5526071e-01-3.7665966e-08j,  3.2785351e-08+1.1740848e+00j,
         3.2212034e-02+4.0837105e-07j],
       [ 3.7284330e-08-1.1740848e+00j,  5.2513175e+00-1.2297018e-08j,
         5.9707546e-09-1.5379127e+00j],
       [ 3.2212041e-02-3.6055735e-07j,  3.3234264e-09+1.5379126e+00j,
         1.0487352e+01+2.2757323e-08j]], dtype=complex64)

# Example using the approximate Transmon implementation 

In [10]:
label = "transmon"
frequency: float = 5.0
anharmonicity: float = -0.3
ext_flux: float = 0.0
dim: int = 5

transmon = AnharmonicOscillator(
    label=label,
    frequency=frequency,
    anharmonicity=anharmonicity,
    ext_flux=ext_flux,
    dim=dim,
)

In [16]:
assert np.allclose(transmon.fundamental_frequency, transmon.frequency)

In [15]:
assert np.allclose(transmon.abs_anharmonicity, transmon.anharmonicity)

In [17]:
print(transmon.get_hamiltonian())

[[ 0.   0.   0.   0.   0. ]
 [ 0.   5.   0.   0.   0. ]
 [ 0.   0.   9.7  0.   0. ]
 [ 0.   0.   0.  14.1  0. ]
 [ 0.   0.   0.   0.  18.2]]


In [8]:
import jax

jax.config.update("jax_enable_x64", True)

In [12]:
from scipy.optimize import minimize

# from jax.scipy.optimize import minimize

# import scipy as sp

import math
from typing import Any, Tuple

label: str = "test"
frequency: float = 5.0
anharmonicity: float = -0.3

ext_flux: float = 0.0
offset_charge: float = 0.0
asymmetry: float = 0.0
charge_cutoff: int = 100

if not isinstance(frequency, float):
    raise ValueError(
        f"The maximum frequency expected to be a float, instead got type {type(frequency)}."
    )
if not isinstance(anharmonicity, float):
    raise ValueError(
        f"The anharmonicity expected to be a float, instead got type {type(anharmonicity)}."
    )
if anharmonicity > 0:
    raise ValueError(
        "The anharmonicity is expected to be negative for a transmon qubits. Instead a positive anharmonicity was provided."
    )

init_ec = -anharmonicity
init_ej = (frequency + init_ec) / (8 * init_ec)

cos_term = math.cos(ext_flux)
sqrt_term = math.sqrt(1 + asymmetry**2 * math.tan(ext_flux) ** 2)
prefactor = abs(cos_term) * sqrt_term
max_ej = init_ej / prefactor

def objective_func(x: Tuple[float, float]) -> float:
    charging_energy, josephson_energy = x
    transmon = TunableTransmon(
        label,
        charging_energy,
        josephson_energy,
        offset_charge,
        ext_flux,
        asymmetry,
        charge_cutoff,
    )
    freq = transmon.fundamental_frequency
    anharm = transmon.abs_anharmonicity
    return (frequency - freq) ** 2 + (anharmonicity - anharm) ** 2

init_guess = (max_ej, init_ec)

result = minimize(objective_func, init_guess)
if not result.success:
    raise ValueError(f"Optimization failed with message: {result.message}.")

print(result)
charging_energy, josephson_energy = result.x

final_transmon = TunableTransmon(
    label,
    charging_energy,
    josephson_energy,
    offset_charge,
    ext_flux,
    asymmetry,
    charge_cutoff,
)

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 2.192270572315795e-13
        x: [ 2.615e-01  1.331e+01]
      nit: 17
      jac: [ 2.735e-07  7.237e-09]
 hess_inv: [[ 2.698e-01 -1.214e+01]
            [-1.214e+01  5.585e+02]]
     nfev: 60
     njev: 20


In [13]:
final_transmon.fundamental_frequency

Array(5.00000001, dtype=float64)

In [14]:
final_transmon.abs_anharmonicity

Array(-0.29999953, dtype=float64)

In [128]:
from scipy.optimize import minimize

# from jax.scipy.optimize import minimize

# import scipy as sp

import math
from typing import Any, Tuple

label: str = "test"
frequency: float = 5.0
anharmonicity: float = -0.3

ext_flux: float = 0.0
offset_charge: float = 0.0
asymmetry: float = 0.0
charge_cutoff: int = 100

if not isinstance(frequency, float):
    raise ValueError(
        f"The maximum frequency expected to be a float, instead got type {type(frequency)}."
    )
if not isinstance(anharmonicity, float):
    raise ValueError(
        f"The anharmonicity expected to be a float, instead got type {type(anharmonicity)}."
    )
if anharmonicity > 0:
    raise ValueError(
        "The anharmonicity is expected to be negative for a transmon qubits. Instead a positive anharmonicity was provided."
    )

init_ec = -anharmonicity
init_ej_eff = (frequency + init_ec) ** 2 / (8 * init_ec)

cos_term = math.cos(ext_flux)
sqrt_term = math.sqrt(1 + asymmetry**2 * math.tan(ext_flux) ** 2)
prefactor = abs(cos_term) * sqrt_term

init_ej = init_ej_eff / prefactor


def objective_func(
    x: Tuple[Any],
    label: str,
    offset_charge: float,
    asymmetry: float,
    ext_flux: float,
    charge_cutoff: int,
) -> float:
    charging_energy, josephson_energy = x
    """
    cos_term = math.cos(ext_flux)
    sqrt_term = math.sqrt(1 + asymmetry**2 * math.tan(ext_flux) ** 2)
    prefactor = abs(cos_term) * sqrt_term
    ej_eff = josephson_energy * prefactor
    
    charge_dim = 2 * charge_cutoff + 1
    charge_vals = np.arange(-charge_cutoff, charge_cutoff + 1)
    n_op = np.diag(charge_vals)
    id_op = np.identity(charge_dim)

    n_offset_op = n_op - offset_charge * id_op
    kinetic_term = 4 * charging_energy * n_offset_op @ n_offset_op

    offdiag_elems = np.ones(2 * charge_cutoff)

    superdiag_mat = np.diag(0.5 * offdiag_elems, 1)
    subdiag_mat = np.transpose(superdiag_mat)
    cosphi_op = superdiag_mat + subdiag_mat

    superdiag_mat = np.diag(0.5j * offdiag_elems, 1)
    subdiag_mat = np.transpose(superdiag_mat)
    sinphi_op = superdiag_mat - subdiag_mat

    phase = math.atan(asymmetry * math.tan(ext_flux))

    cos_term = cosphi_op * math.cos(phase)
    sin_term = sinphi_op * math.sin(phase)

    potential_term = -ej_eff * (cos_term + sin_term)
    
    hamiltonian = kinetic_term + potential_term
    """

    transmon = TunableTransmon(
        label=label,
        charging_energy=charging_energy,
        josephson_energy=josephson_energy,
        offset_charge=offset_charge,
        ext_flux=ext_flux,
        asymmetry=asymmetry,
        charge_cutoff=charge_cutoff,
    )
    """
    hamiltonian = transmon.get_hamiltonian()

    eig_vals, _ = jnp.linalg.eigh(hamiltonian)

    freq = eig_vals[1] - eig_vals[0]
    anharm = eig_vals[2] - (2 * eig_vals[1])  + eig_vals[0]
    
    """
    freq = transmon.fundamental_frequency
    anharm = transmon.abs_anharmonicity
    print(x, freq, anharm)
    return (frequency - freq) ** 2 + (anharmonicity - anharm) ** 2


init_guess = [init_ec, init_ej]

ej_bounds = (0.0, 100.0)
ec_bounds = (0.0, 10.0)
bounds = (ec_bounds, ej_bounds)

result = minimize(
    objective_func,
    init_guess,
    bounds=bounds,
    args=(label, charge_cutoff, offset_charge, asymmetry, ext_flux),
)

print(result)
charging_energy, josephson_energy = result.x

final_transmon = TunableTransmon(
    label,
    charging_energy,
    josephson_energy,
    offset_charge,
    ext_flux,
    asymmetry,
    charge_cutoff,
)

ValueError: The offset charge expected to be a float, instead got type <class 'int'>.

In [126]:
final_transmon.abs_anharmonicity

Array(-0.29999977, dtype=float64)

In [127]:
final_transmon.fundamental_frequency

Array(5.00000017, dtype=float64)

In [90]:
freq = frequency
anharm = anharmonicity
phi_ext = ext_flux
d = asymmetry
print_optim = True

import numpy as np


def compute_transmon_levels_numpy(
    EJ_GHz: float = 18.02943828,
    EC_GHz: float = 0.2400841,
    ng: float = 0.0,
    dim: int = 5,
    n_charge_states: int = 100,
):
    dtype = complex
    ### Building the operators in the charge basis ###
    charge_op = np.diag(np.arange(-1 * n_charge_states, n_charge_states + 1))
    dim_charge = 2 * n_charge_states + 1
    ones_offdiag = np.ones((dim_charge - 1,), dtype=dtype)
    cosphi = np.diag(0.5 * ones_offdiag, k=1) + np.diag(0.5 * ones_offdiag, k=-1)
    sinphi = np.diag(0.5j * ones_offdiag, k=1) + np.diag(-0.5j * ones_offdiag, k=-1)

    ### Building the Hamiltonian ###
    I = np.identity(dim_charge)
    # H = 4 Ec (n - ng)^2 - Ej cos(phi)
    H = 4 * EC_GHz * (charge_op - ng * I) @ (charge_op - ng * I) - EJ_GHz * cosphi
    # Do the diagonalization
    [D, V] = np.linalg.eigh(H)
    eigvals = np.array(D[:dim] - D[0], dtype=dtype)
    return eigvals.real


EC_guess = -anharm
EJ_guess = (freq + EC_guess) ** 2 / (8 * EC_guess)
EJ_max_guess = float(
    EJ_guess / (jnp.abs(jnp.cos(phi_ext)) * jnp.sqrt(1 + d**2 * jnp.tan(phi_ext) ** 2))
)


def cost_fn(x):
    EJ, EC = x
    energies = compute_transmon_levels_numpy(
        EJ_GHz=EJ, EC_GHz=EC, dim=3, n_charge_states=100
    )
    freq_ = energies[1]
    anharm_ = energies[2] - 2 * freq_
    print(x, freq_, anharm_)
    return (freq - freq_) ** 2 + (anharm - anharm_) ** 2


result = minimize(
    cost_fn, x0=[EJ_max_guess, EC_guess], bounds=[(0.0, 100.0), (0.0, 10.0)]
)
if print_optim:
    print(result)

final_transmon = TunableTransmon(
    label,
    result.x[1],
    result.x[0],
    offset_charge,
    ext_flux,
    asymmetry,
    charge_cutoff,
)

print(final_transmon.fundamental_frequency)
print(final_transmon.abs_anharmonicity)

[11.70416641  0.3       ] 4.979840703488856 -0.35423027882179525
[11.70416642  0.3       ] 4.979840705763146 -0.35423027878198
[11.70416641  0.30000001] 4.979840780754658 -0.35423029218289415
[11.71376787  0.466609  ] 6.105822377881005 -0.6081064989460074
[11.71376788  0.466609  ] 6.105822380720429 -0.6081064987551184
[11.71376787  0.46660901] 6.105822437455205 -0.6081065167705155
[11.70424939  0.30143985] 4.990969326783944 -0.35615532565129193
[11.7042494   0.30143985] 4.9909693290637245 -0.3561553256109047
[11.70424939  0.30143986] 4.9909694038358925 -0.3561553390343324
[11.70428394  0.30135077] 4.990290726829092 -0.356035969700061
[11.70428395  0.30135077] 4.9902907291085326 -0.35603596965971285
[11.70428394  0.30135078] 4.990290803894381 -0.356035983081723
[11.70433888  0.30133098] 4.9901507249357495 -0.3560092641472892
[11.70433889  0.30133098] 4.990150727215107 -0.35600926410695166
[11.70433888  0.30133099] 4.990150802004181 -0.3560092775286243
[11.70497435  0.30120869] 4.9893529

In [85]:
n_charge_states = 100
ec = final_transmon.charging_energy
ej = final_transmon.max_josephson_energy

charge_op = np.diag(np.arange(-1 * n_charge_states, n_charge_states + 1))
dim_charge = 2 * n_charge_states + 1
ones_offdiag = np.ones((dim_charge - 1,))
cosphi = np.diag(0.5 * ones_offdiag, k=1) + np.diag(0.5 * ones_offdiag, k=-1)
sinphi = np.diag(0.5j * ones_offdiag, k=1) + np.diag(-0.5j * ones_offdiag, k=-1)

### Building the Hamiltonian ###
I = np.identity(dim_charge)
# H = 4 Ec (n - ng)^2 - Ej cos(phi)
H = (
    4 * ec * (charge_op - offset_charge * I) @ (charge_op - offset_charge * I)
    - ej * cosphi
)
# Do the diagonalization
[D, V] = np.linalg.eigh(H)

In [99]:
np.allclose(D[:6], np.array(final_transmon.eigenvalues())[:6])

True

In [97]:
D[:3]

array([-10.73880098,  -5.7388008 ,  -1.0388004 ])

In [96]:
final_transmon.eigenvalues()

Array([-1.0738800e+01, -5.7388000e+00, -1.0388013e+00,  3.3262057e+00,
        7.2564917e+00,  1.0950668e+01,  1.2919883e+01,  1.8103233e+01,
        1.8241119e+01,  2.7022493e+01,  2.7023193e+01,  3.8250607e+01,
        3.8250614e+01,  5.1685390e+01,  5.1685394e+01,  6.7270210e+01,
        6.7270218e+01,  8.4979866e+01,  8.4979889e+01,  1.0480162e+02,
        1.0480163e+02,  1.2672831e+02,  1.2672832e+02,  1.5075568e+02,
        1.5075568e+02,  1.7688109e+02,  1.7688109e+02,  2.0510286e+02,
        2.0510286e+02,  2.3541969e+02,  2.3541972e+02,  2.6783087e+02,
        2.6783087e+02,  3.0233585e+02,  3.0233588e+02,  3.3893408e+02,
        3.3893408e+02,  3.7762534e+02,  3.7762534e+02,  4.1840927e+02,
        4.1840930e+02,  4.6128592e+02,  4.6128598e+02,  5.0625504e+02,
        5.0625510e+02,  5.5331635e+02,  5.5331641e+02,  6.0246997e+02,
        6.0247003e+02,  6.5371576e+02,  6.5371576e+02,  7.0705365e+02,
        7.0705371e+02,  7.6248352e+02,  7.6248358e+02,  8.2000555e+02,
      

In [36]:
def compute_transmon_levels(
    EJ_GHz: float = 18.02943828,
    EC_GHz: float = 0.2400841,
    ng: float = 0.0,
    dim: int = 5,
    n_charge_states: int = 100,
):
    dtype = jnp.complex64
    ### Building the operators in the charge basis ###
    charge_op = jnp.diag(jnp.arange(-1 * n_charge_states, n_charge_states + 1))
    dim_charge = 2 * n_charge_states + 1
    ones_offdiag = jnp.ones((dim_charge - 1,), dtype=dtype)
    cosphi = jnp.diag(0.5 * ones_offdiag, k=1) + jnp.diag(0.5 * ones_offdiag, k=-1)
    sinphi = jnp.diag(0.5j * ones_offdiag, k=1) + jnp.diag(-0.5j * ones_offdiag, k=-1)

    ### Building the Hamiltonian ###
    I = jnp.identity(dim_charge)
    # H = 4 Ec (n - ng)^2 - Ej cos(phi)
    H = 4 * EC_GHz * (charge_op - ng * I) @ (charge_op - ng * I) - EJ_GHz * cosphi
    # Do the diagonalization
    [D, V] = jnp.linalg.eigh(H)
    eigvals = jnp.array(D[:dim] - D[0], dtype=dtype)
    return eigvals.real

In [43]:
freq = frequency
anharm = anharmonicity
phi_ext = ext_flux
d = asymmetry
print_optim = True

import numpy as np


def compute_transmon_levels_numpy(
    EJ_GHz: float = 18.02943828,
    EC_GHz: float = 0.2400841,
    ng: float = 0.0,
    dim: int = 5,
    n_charge_states: int = 100,
):
    dtype = complex
    ### Building the operators in the charge basis ###


EC_guess = -anharm
EJ_guess = (freq + EC_guess) ** 2 / (8 * EC_guess)
EJ_max_guess = float(
    EJ_guess / (jnp.abs(jnp.cos(phi_ext)) * jnp.sqrt(1 + d**2 * jnp.tan(phi_ext) ** 2))
)
n_charge_states = 100
ng = 0.0


def cost_fn(x):
    charge_op = np.diag(np.arange(-1 * n_charge_states, n_charge_states + 1))

    dim_charge = 2 * n_charge_states + 1
    ones_offdiag = np.ones((dim_charge - 1,), dtype=jnp.complex64)
    cosphi = np.diag(0.5 * ones_offdiag, k=1) + np.diag(0.5 * ones_offdiag, k=-1)
    ### Building the Hamiltonian ###
    I = np.identity(dim_charge)
    # H = 4 Ec (n - ng)^2 - Ej cos(phi)
    H = 4 * x[0] * (charge_op - ng * I) @ (charge_op - ng * I) - x[1] * cosphi
    # Do the diagonalization
    eig_vals, _ = np.linalg.eigh(H)

    freq_ = eig_vals[1] - eig_vals[0]
    anharm_ = eig_vals[2] - 2 * eig_vals[1] + eig_vals[0]
    return (freq - freq_) ** 2 + (anharm - anharm_) ** 2


solver = ScipyMinimize(fun=cost_fn, method="BFGS")
sol = solver.run(init_guess)

if print_optim:
    print(result)

TracerArrayConversionError: The numpy.ndarray conversion method __array__() was called on traced array with shape complex64[201,201].
See https://jax.readthedocs.io/en/latest/errors.html#jax.errors.TracerArrayConversionError

In [25]:
from typing import Union, Callable

Numeric = Union[float, complex]
GenNumeric = Union[Numeric, Callable]

In [27]:
def test_func(x, y):
    return x + y
isinstance(test_func, GenNumeric)

True